## Import Libraries

In [1]:
import os
import requests

API_TOKEN = os.environ.get("FOOTBALL_DATA_TOKEN")

if not API_TOKEN:
    raise ValueError("Token not found in environment variables.")

headers = {"X-Auth-Token": API_TOKEN}

leagues = {
    "PD": "LaLiga",
    "PPL": "Primeira Liga"
}

for code, name in leagues.items():
    url = f"https://api.football-data.org/v4/competitions/{code}"
    response = requests.get(url, headers=headers)
    print(f"{name} ({code}) -> Status code: {response.status_code}")

LaLiga (PD) -> Status code: 200
Primeira Liga (PPL) -> Status code: 200


## Function to get matches from the API

This function asks the API for all matches of one league and one season.

In [2]:
def fetch_matches(competition_code, season, token):
    """
    Fetch all matches for a competition/season from football-data.org v4.

    Args:
        competition_code (str): e.g. "PD" for LaLiga, "PPL" for Primeira Liga
        season (int): starting year of the season, e.g. 2023 for 2023-2024
        token (str): API token

    Returns:
        list[dict]: raw match records as returned by the API
    """
    url = f"https://api.football-data.org/v4/competitions/{competition_code}/matches"
    headers = {"X-Auth-Token": token}
    params = {"season": season}

    response = requests.get(url, headers=headers, params=params)

    if response.status_code != 200:
        raise RuntimeError(
            f"Request failed for {competition_code} season {season}: "
            f"status {response.status_code} - {response.text}"
        )

    return response.json()["matches"]


# Quick test: fetch LaLiga 2023-2024 and check how many matches came back
test_matches = fetch_matches("PD", 2023, API_TOKEN)
print(f"LaLiga 2023-2024: {len(test_matches)} matches retrieved")

LaLiga 2023-2024: 380 matches retrieved


### Get 3 past seasons for both leagues

This loop calls the function above 6 times: 2 leagues x 3 seasons.
We wait a few seconds between calls so the API does not block us.

In [3]:
import time

leagues = {"PD": "LaLiga", "PPL": "Primeira_Liga"}
seasons = [2023, 2024, 2025]  

raw_matches = {}

for code, name in leagues.items():
    for season in seasons:
        matches = fetch_matches(code, season, API_TOKEN)
        raw_matches[(code, season)] = matches
        print(f"{name} ({code}) {season}-{season + 1}: {len(matches)} matches retrieved")
        time.sleep(6) 

LaLiga (PD) 2023-2024: 380 matches retrieved
LaLiga (PD) 2024-2025: 380 matches retrieved
LaLiga (PD) 2025-2026: 380 matches retrieved
Primeira_Liga (PPL) 2023-2024: 306 matches retrieved
Primeira_Liga (PPL) 2024-2025: 306 matches retrieved
Primeira_Liga (PPL) 2025-2026: 306 matches retrieved


## Get the current season (2026-2027)

We keep this separate from the past seasons because this season is not finished yet. New matches will be added every week.

In [4]:
active_season = 2026
active_matches = {}

for code, name in leagues.items():
    matches = fetch_matches(code, active_season, API_TOKEN)
    active_matches[code] = matches
    print(f"{name} ({code}) {active_season}-{active_season + 1}: {len(matches)} matches retrieved")
    time.sleep(6)

LaLiga (PD) 2026-2027: 380 matches retrieved
Primeira_Liga (PPL) 2026-2027: 306 matches retrieved


## Check match status

A match can be already played or not played yet. This cell counts how many matches are in each status, to check the data looks right.

In [5]:
# Check how many matches are finished vs scheduled in the active season
from collections import Counter

for code, name in leagues.items():
    statuses = Counter(m["status"] for m in active_matches[code])
    print(f"{name} ({code}): {dict(statuses)}")

LaLiga (PD): {'FINISHED': 69, 'TIMED': 31, 'SCHEDULED': 280}
Primeira_Liga (PPL): {'FINISHED': 62, 'POSTPONED': 1, 'TIMED': 45, 'SCHEDULED': 198}


### Turn API data into flat rows that match our schema

The API sends nested data (data inside data). This function flattens each match into the exact column names and types our bronze schema expects, so it is ready to become a Parquet file.

In [6]:
import datetime as dt

# Map our short API codes to the readable league names used in the schema
LEAGUE_NAME_MAP = {
    "PD": "laliga",
    "PPL": "primeira_liga",
}


def map_match_to_bronze(match, competition_code, season_start_year):
    """
    Turn one raw match record from the API into a flat dictionary
    that matches the columns and types in MATCHES_BRONZE_SCHEMA.

    Score fields are None for matches not played yet (status other
    than FINISHED) - this is expected for the active season.
    """
    score = match.get("score", {}).get("fullTime", {})
    season_str = f"{season_start_year}-{season_start_year + 1}"
    match_date = dt.datetime.fromisoformat(match["utcDate"].replace("Z", "+00:00"))

    return {
        "league": LEAGUE_NAME_MAP[competition_code],
        "season": season_str,
        "matchday": match.get("matchday"),
        "match_date": match_date,
        "home_team": match["homeTeam"]["name"],
        "away_team": match["awayTeam"]["name"],
        "home_team_id": str(match["homeTeam"]["id"]),
        "away_team_id": str(match["awayTeam"]["id"]),
        "home_score": score.get("home"),
        "away_score": score.get("away"),
        "status": match["status"],
        "home_xg": None,   # not available on the free tier
        "away_xg": None,   # not available on the free tier
        "source": "football_data_org",
    }


# Quick test: map the historical LaLiga 2023 matches and inspect the first row
mapped_test = [map_match_to_bronze(m, "PD", 2023) for m in raw_matches[("PD", 2023)]]
print(mapped_test[0])

{'league': 'laliga', 'season': '2023-2024', 'matchday': 1, 'match_date': datetime.datetime(2023, 8, 11, 17, 30, tzinfo=datetime.timezone.utc), 'home_team': 'UD Almería', 'away_team': 'Rayo Vallecano de Madrid', 'home_team_id': '267', 'away_team_id': '87', 'home_score': 0, 'away_score': 2, 'status': 'FINISHED', 'home_xg': None, 'away_xg': None, 'source': 'football_data_org'}


### Project paths setup

We define where the project lives on disk. LAKE_ROOT is the folder where all Parquet files (bronze, silver, gold) will be saved.

In [7]:
from pathlib import Path

# Root folder of the project on this device
PROJECT_ROOT = Path.cwd()
LAKE_ROOT = PROJECT_ROOT / "data"

print(f"Project root exists: {PROJECT_ROOT.exists()}")
print(f"Lake root exists: {LAKE_ROOT.exists()}")
print(f"Contents of PROJECT_ROOT: {sorted(p.name for p in PROJECT_ROOT.iterdir())}")

Project root exists: True
Lake root exists: True
Contents of PROJECT_ROOT: ['Football_Data_Extraction.ipynb', 'Match_Predictions_Builder.ipynb', 'Matches_Silver_Builder.ipynb', 'Season_Simulations_Builder.ipynb', 'Team_Ratings_Builder.ipynb', 'Teams_Master_Builder.ipynb', 'Tokens.txt', 'complete_teams_master.py', 'config', 'data', 'setup_data_lake.py', 'src']


### Write matches to the bronze layer as Parquet

We turn each league/season into a table with our fixed schema, then save it as one Parquet file per league and season, using the write_partition function we already built in parquet_io.py.

In [8]:
import sys
sys.path.append(str(PROJECT_ROOT / "src"))

from utils.schema import MATCHES_BRONZE_SCHEMA
from utils.parquet_io import write_partition, read_league_season, read_league_all_seasons
import pyarrow as pa


def write_season_to_bronze(matches_raw, competition_code, season_start_year):
    """
    Map and write one league/season of matches to the bronze layer.
    Returns the path of the file written, for logging.
    """
    league_name = LEAGUE_NAME_MAP[competition_code]
    season_str = f"{season_start_year}-{season_start_year + 1}"

    rows = [map_match_to_bronze(m, competition_code, season_start_year) for m in matches_raw]
    table = pa.Table.from_pylist(rows, schema=MATCHES_BRONZE_SCHEMA)

    output_path = LAKE_ROOT / "bronze" / league_name / f"season={season_str}" / "matches.parquet"
    write_partition(table, output_path)

    print(f"{league_name} {season_str}: {table.num_rows} rows written to {output_path.relative_to(PROJECT_ROOT)}")
    return output_path


# Write the 3 historical seasons for both leagues
for code in leagues:
    for season in seasons:
        write_season_to_bronze(raw_matches[(code, season)], code, season)

# Write the active season separately (this file gets overwritten on future refreshes)
for code in leagues:
    write_season_to_bronze(active_matches[code], code, active_season)

laliga 2023-2024: 380 rows written to data\bronze\laliga\season=2023-2024\matches.parquet
laliga 2024-2025: 380 rows written to data\bronze\laliga\season=2024-2025\matches.parquet
laliga 2025-2026: 380 rows written to data\bronze\laliga\season=2025-2026\matches.parquet
primeira_liga 2023-2024: 306 rows written to data\bronze\primeira_liga\season=2023-2024\matches.parquet
primeira_liga 2024-2025: 306 rows written to data\bronze\primeira_liga\season=2024-2025\matches.parquet
primeira_liga 2025-2026: 306 rows written to data\bronze\primeira_liga\season=2025-2026\matches.parquet
laliga 2026-2027: 380 rows written to data\bronze\laliga\season=2026-2027\matches.parquet
primeira_liga 2026-2027: 306 rows written to data\bronze\primeira_liga\season=2026-2027\matches.parquet


## Read back and check the bronze files

This reads back the Parquet files we just wrote and checks that the row counts still match what we got from the API earlier.

In [9]:
# Leagues and seasons we expect to find in the bronze layer
layer = "bronze"
league_names = ["laliga", "primeira_liga"]
season_strs = ["2023-2024", "2024-2025", "2025-2026"]
active_season_str = "2026-2027"

# Row counts we already confirmed from the API
expected_historical_rows = {
    "laliga": 380,
    "primeira_liga": 306,
}

# Check each historical season file
print("Historical seasons:")
for league_name in league_names:
    for season_str in season_strs:
        table_season = read_league_season(LAKE_ROOT, layer, league_name, season_str)
        expected = expected_historical_rows[league_name]
        status = "OK" if table_season.num_rows == expected else "MISMATCH"
        print(f"  {league_name} {season_str}: {table_season.num_rows} rows (expected {expected}) -> {status}")

# Check the active season file and its status mix
print("\nActive season (2026-2027):")
for league_name in league_names:
    table_active = read_league_season(LAKE_ROOT, layer, league_name, active_season_str)
    df_active = table_active.to_pandas()
    print(f"  {league_name}: {len(df_active)} rows total")
    print(df_active["status"].value_counts().to_string())

# Check that reading all seasons together also works
print("\nAll seasons combined:")
for league_name in league_names:
    table_all = read_league_all_seasons(LAKE_ROOT, layer, league_name)
    print(f"  {league_name}: {table_all.num_rows} rows total")

Historical seasons:
  laliga 2023-2024: 380 rows (expected 380) -> OK
  laliga 2024-2025: 380 rows (expected 380) -> OK
  laliga 2025-2026: 380 rows (expected 380) -> OK
  primeira_liga 2023-2024: 306 rows (expected 306) -> OK
  primeira_liga 2024-2025: 306 rows (expected 306) -> OK
  primeira_liga 2025-2026: 306 rows (expected 306) -> OK

Active season (2026-2027):
  laliga: 380 rows total
status
SCHEDULED    280
FINISHED      69
TIMED         31
  primeira_liga: 306 rows total
status
SCHEDULED    198
FINISHED      62
TIMED         45
POSTPONED      1

All seasons combined:
  laliga: 1520 rows total
  primeira_liga: 1224 rows total


## Diagnostic odd status values in LaLiga active season

Primeira Liga status values came back clean, but LaLiga active season has 31 matches where status is a raw date-time string instead of a normal status code. This cell inspects one of them before we treat this notebook as fully closed.


In [10]:
# Diagnostic: inspect the odd status values found in LaLiga active season
# (status field showing raw date-time strings instead of a normal status code)
raw_status_values = set(m["status"] for m in active_matches["PD"])
KNOWN_STATUSES = {"SCHEDULED", "TIMED", "LIVE", "IN_PLAY", "PAUSED",
                  "FINISHED", "SUSPENDED", "POSTPONED", "CANCELED", "AWARDED"}
odd = sorted(v for v in raw_status_values if v not in KNOWN_STATUSES)
print(f"Odd status values found: {len(odd)}")
print(odd[:5])

if odd:
    sample = next(m for m in active_matches["PD"] if m["status"] == odd[0])
    print("\nFull raw record for one of these matches:")
    print(sample)


Odd status values found: 0
[]
